# Accessing satellite data using the Copernicus Marine Toolbox

Before we begin, we need to install the ```copernicusmarine``` toolbox in colab:

In [ ]:
!pip install copernicusmarine

Now we can start working with the toolbox. First we import it, then we can check the version to make sure that the library is available as expected.

In [ ]:
import copernicusmarine as cm

In [ ]:
cm.__version__

## Logging in

Before we can start accessing data, we need to log in. In the username="replace_with_your_username" command below, replace the "replace_with_your_username" string with your login name (check your e-mail from Copernicus for this information)

You will then be prompted to enter your password. It is the same one that you used to log in to the Copernicus Marine website.

In [ ]:
cm.login(username="replace_with_your_username")

## Accessing data

We can access data a number of ways using the toolbox.

We first consider the case where we use the interface to download a file.

To do this, we can use the ```subset``` command of the library. As usual in python, you can accessing the documentation by using ```?``` after the command:

In [ ]:
cm.subset?

Let's look at an example. We'll download SST data from [the C3S global Sea Surface and Sea Ice Temperature Reprocessed product](https://data.marine.copernicus.eu/product/SST_GLO_SST_L4_REP_OBSERVATIONS_010_024/description), for 12th March 2020, over the region 30-45°S, 10°W-35°E. We will save the file in the current working directory, and we will choose the output filename.

In [ ]:
cm.subset(
    dataset_id= "C3S-GLO-SST-L4-REP-OBS-SST",# string
    variables= ["analysed_sst"], # list of strings
    start_datetime= "2020-03-12T00:00:00", #string
    end_datetime= "2020-03-12T23:59:59", #string
    minimum_longitude= -10, # float
    maximum_longitude= 35, # float
    minimum_latitude= -45, # float
    maximum_latitude= -30, # float
    output_directory= "./", # string
    output_filename= "sst_20200312.nc", # string
)

Click on the little folder icon in the toolbar on the left side of your screen. You will see the current working directory. Within it, there should be a file with the output name that you specified in the last cell.

Now it's your turn. Try downloading chlorophyll-A data from the [Global Ocean Colour (Copernicus-GlobColour)](https://data.marine.copernicus.eu/product/OCEANCOLOUR_GLO_BGC_L4_NRT_009_102/description) product for the period 13th-15th May 2022, over the region: 5°N-5°S, 60°W-30°E. You should retrieve two variables: "Mass concentration of chlorophyll a in sea water" and "Standard deviation of mass concentration of chlorophyll a in sea water". You can fill in the code below:

In [ ]:
cm.subset(
    dataset_id= ,# string
    variables= , # list of strings
    start_datetime= , #string
    end_datetime= , #string
    minimum_longitude= , # float
    maximum_longitude= , # float
    minimum_latitude= , # float
    maximum_latitude= , # float
    output_directory= , # string
    output_filename= , # string
)

Once we have downloaded this data, we can then analyse it using whichever library we prefer.

Let's use ```xarray``` to plot the SST data that we downloaded earlier:

In [ ]:
import xarray as xr
ds = xr.open_dataset("sst_20200312.nc")
ds

We can see the variable: ```analysed_sst``` and the metadata that is associated with it.

We can make a plot for the time step that we downloaded directly using ```xarray```:

In [ ]:
ds['analysed_sst'].isel(time=0).plot()

### Loading directly into an xarray data set

We do not have to save directly to a file when we access data using the toolbox: we can choose instead to load the data into an xarray data set. This can be useful is we want to, for example, average the data before saving it. Let's use the same SST example as before and load the data directly using the ```open_dataset``` command.

You can see that the subsetting information that we give is the same as for the ```subset``` command that we used previously. However, this time we do not need to give information about the output directory or file name, because the ```open_dataset``` command will not try to create a file for us: we will have to do that ourselves when we are ready.

In [ ]:
ds = cm.open_dataset(
    dataset_id= "C3S-GLO-SST-L4-REP-OBS-SST",# string
    variables= ["analysed_sst"], # list of strings
    start_datetime= "2020-03-12T00:00:00", #string
    end_datetime= "2020-03-12T23:59:59", #string
    minimum_longitude= -10, # float
    maximum_longitude= 30, # float
    minimum_latitude= -45, # float
    maximum_latitude= -30, # float
)

In [ ]:
ds

In [ ]:
ds['analysed_sst'].isel(time=0).plot()

As an example of when this command might be really useful, let's download 10 days of data and make a 10-day mean:

In [ ]:
ds_10 = cm.open_dataset(
    dataset_id= "C3S-GLO-SST-L4-REP-OBS-SST",# string
    variables= ["analysed_sst"], # list of strings
    start_datetime= "2020-03-12T00:00:00", #string
    end_datetime= "2020-03-21T23:59:59", #string
    minimum_longitude= -10, # float
    maximum_longitude= 30, # float
    minimum_latitude= -45, # float
    maximum_latitude= -30, # float
)

In [ ]:
ds_10

Making the 10-day average is now really easy:

In [ ]:
ds_mean = ds_10.mean(dim='time')
ds_mean

And we can now save this directly, without having to download the 10 individual daily files:

In [ ]:
ds_mean.to_netcdf('./sst_10_day_mean.nc')

You should see the new file appear in your file browser on the left of the screen.

It can be a good idea to think about whether we have sufficient disk and memory space to treat the data that we want to use before we try to download it!

We can check the data size of a request in the Copernicus Marine toolbox by adding the line: ```dry_run=True``` to our request. Let's do that for the SST example again. It works only for the ```subset``` command:

In [ ]:
cm.subset(
    dataset_id= "C3S-GLO-SST-L4-REP-OBS-SST",# string
    variables= ["analysed_sst"], # list of strings
    start_datetime= "2020-03-12T00:00:00", #string
    end_datetime= "2020-03-12T23:59:59", #string
    minimum_longitude= -10, # float
    maximum_longitude= 30, # float
    minimum_latitude= -45, # float
    maximum_latitude= -30, # float
    dry_run=True
);

We see that the estimated size of the download here is small.

Try modifying the command for the [Global Ocean Colour (Copernicus-GlobColour)](https://data.marine.copernicus.eu/product/OCEANCOLOUR_GLO_BGC_L4_NRT_009_102/description) product for the period 13th-15th May 2022, over the region: 5°N-5°S, 60°W-30°E. This is the same command that you used before, but this time we only want to check the data size.

In [ ]:
# enter your code here
cm.subset(
    dataset_id="c3s_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M",# string
    variables=["CHL","CHL_error"] , # list of strings
    start_datetime= "13-05-2022", #string
    end_datetime= "15-05-2022", #string
    minimum_longitude= -60, # float
    maximum_longitude= 30, # float
    minimum_latitude= -5, # float
    maximum_latitude= 5, # float
    dry_run=True
);

Finally, check what the file size would be if you wanted to download 1 year of data for the same variables over the full global region (90°N-90°S, 180°W-180°E). Check for the January - December of the year 2022:

In [ ]:
# enter your code here
cm.subset(
    dataset_id="c3s_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M",# string
    variables=["CHL","CHL_error"] , # list of strings
    start_datetime= "01-01-2022", #string
    end_datetime= "31-12-2022", #string
    minimum_longitude= -180, # float
    maximum_longitude= 180, # float
    minimum_latitude= -90, # float
    maximum_latitude= 90, # float
    dry_run=True
);

# Plotting data

We have already seen that we can make maps of data using ```xarray```'s `.plot()` command. We can also use this to plot time series at a given point if have downloaded multiple time steps.

Let's look at our SST example again. We'll download 1 year of data over our region this time. Let's check the file size first:

In [ ]:
cm.subset(
    dataset_id= "C3S-GLO-SST-L4-REP-OBS-SST",# string
    variables= ["analysed_sst"], # list of strings
    start_datetime= "2020-01-01", #string
    end_datetime= "2020-12-31", #string
    minimum_longitude= -10, # float
    maximum_longitude= 30, # float
    minimum_latitude= -45, # float
    maximum_latitude= -30, # float
    dry_run=True
);

Now we'll fetch the data:

In [ ]:
ds = cm.open_dataset(
    dataset_id= "C3S-GLO-SST-L4-REP-OBS-SST",# string
    variables= ["analysed_sst"], # list of strings
    start_datetime= "2020-01-01", #string
    end_datetime= "2020-12-31", #string
    minimum_longitude= -10, # float
    maximum_longitude= 30, # float
    minimum_latitude= -45, # float
    maximum_latitude= -30, # float
)

In [ ]:
ds

We can see that we have 366 time steps. Let's look at the time series for the point nearest to the position: 5°E, 35°S. We can select the point using the `.sel` command, and then use `.plot` to plot the time series as before:

In [ ]:
# isolate an individual spatial point in the data set
ds.sel(latitude=-35,longitude=5,method='nearest')

In [ ]:
# same as above, but now we also plot it:
ds['analysed_sst'].sel(latitude=-35,longitude=5,method='nearest').plot()

Note also that with the `.plot` command, we can use many of the same arguments that we would use with `matplotlib`, for example we can use `vmin` and `vmax` to specify limits on the colour scale, or we can specify the colour map with `cmap`:

In [ ]:
ds['analysed_sst'].sel(time="2020-03-15",method='nearest').plot(cmap='inferno')

And finally, we can also extract the numpy arrays from the xarray data set using the `.values` method. This allows you to calculate with the values, or plot them using `matplotlib` if you prefer. We can either extract all of the values, or a subset into numpy arrays. For example:

In [ ]:
# first let's extract all of the latitude data as a numpy array
lat = ds['latitude'].values
print(lat.shape)

In [ ]:
print(lat)

In [ ]:
# now let's extract just a subset of the data:
# we'll choose latitude values between 30-40°S

# to choose the subset range, we can use the slice command:
ds.sel(latitude=slice(-40,-30))

In [ ]:
# and as before, we can extract the numpy values by specifying
# our chosen variable after ds, then adding .values at the end:
lat_subset = ds['latitude'].sel(latitude=slice(-40,-30)).values
print(lat_subset.size)

In [ ]:
print(lat_subset)